# 6CS012 - Worksheet 6: Image Classification with CNN and Transfer Learning
## Complete Solution — Task 1 & Task 2
---

## Imports

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from PIL import Image, UnidentifiedImageError

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Flatten, Dense,
    BatchNormalization, Dropout, Activation,
    GlobalAveragePooling2D
)
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.applications import VGG16
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report

print("TensorFlow version:", tf.__version__)
print("Keras version:", keras.__version__)

---
## Section 2.1 — Data Understanding and Visualization

In [ ]:
# ── Set your dataset path here ──────────────────────────────────────────────
# Update 'train_dir' to point to the FruitsInAmazon train folder on your machine
train_dir = "path/to/FruitsInAmazon/train"   # <-- CHANGE THIS
# ────────────────────────────────────────────────────────────────────────────

# 1. Read the dataset directory and extract class names
class_names = sorted(os.listdir(train_dir))
if not class_names:
    print("No class directories found in the train folder!")
else:
    print(f"Found {len(class_names)} classes: {class_names}")

In [ ]:
# 2. Check for corrupted images
corrupted_images = []

for class_name in class_names:
    class_path = os.path.join(train_dir, class_name)
    if os.path.isdir(class_path):
        for img_name in os.listdir(class_path):
            img_path = os.path.join(class_path, img_name)
            try:
                with Image.open(img_path) as img:
                    img.verify()
            except (IOError, UnidentifiedImageError):
                corrupted_images.append(img_path)

if corrupted_images:
    print("Corrupted Images Found:")
    for img in corrupted_images:
        print(img)
        os.remove(img)   # Remove corrupted images
    print(f"\nRemoved {len(corrupted_images)} corrupted image(s).")
else:
    print("No corrupted images found.")

In [ ]:
# 3. Count class balance
class_counts = {}
for class_name in class_names:
    class_path = os.path.join(train_dir, class_name)
    if os.path.isdir(class_path):
        images = [
            img for img in os.listdir(class_path)
            if img.lower().endswith(('.png', '.jpg', '.jpeg'))
        ]
        class_counts[class_name] = len(images)

print("\nClass Distribution:")
print("=" * 45)
print(f"{'Class Name':<25}{'Valid Image Count':>15}")
print("=" * 45)
for class_name, count in class_counts.items():
    print(f"{class_name:<25}{count:>15}")
print("=" * 45)

# Bar chart for class distribution
plt.figure(figsize=(8, 4))
plt.bar(class_counts.keys(), class_counts.values(), color='steelblue')
plt.title('Class Distribution')
plt.xlabel('Class')
plt.ylabel('Number of Images')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# 4. Visualize one random image per class
selected_images = []
selected_labels = []

for class_name in class_names:
    class_path = os.path.join(train_dir, class_name)
    if os.path.isdir(class_path):
        images = [
            img for img in os.listdir(class_path)
            if img.lower().endswith(('.png', '.jpg', '.jpeg'))
        ]
        if images:
            selected_img = os.path.join(class_path, random.choice(images))
            selected_images.append(selected_img)
            selected_labels.append(class_name)

num_classes = len(selected_images)
cols = (num_classes + 1) // 2
rows = 2

fig, axes = plt.subplots(rows, cols, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    if i < num_classes:
        img = mpimg.imread(selected_images[i])
        ax.imshow(img)
        ax.set_title(selected_labels[i])
        ax.axis('off')
    else:
        ax.axis('off')
plt.suptitle('One Random Image Per Class', fontsize=14)
plt.tight_layout()
plt.show()

---
## Section 2.2 — Data Generation and Pre-processing

In [ ]:
# Hyperparameters
IMAGE_SIZE  = (224, 224)   # VGG16 expects 224×224; same used for Task 1 for consistency
BATCH_SIZE  = 32
NUM_CLASSES = len(class_names)   # 6 for FruitsInAmazon

# Generate train / validation datasets (80 / 20 split)
train_ds, val_ds = keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="both",
    seed=1337,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
)

# Inspect one batch
for images, labels in train_ds.take(1):
    print("Images shape:", images.shape)
    print("Labels shape:", labels.shape)

In [ ]:
# Visualise 9 training images
plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(np.array(images[i]).astype('uint8'))
        plt.title(class_names[int(labels[i])])
        plt.axis('off')
plt.suptitle('Sample Training Images', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── Data Augmentation (New Keras API) ────────────────────────────────────────
data_augmentation_layers = [
    layers.RandomFlip("horizontal"),
    layers.RandomFlip("vertical"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.15),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomContrast(0.1),
]

def data_augmentation(images):
    for layer in data_augmentation_layers:
        images = layer(images)
    return images

# Visualise augmented images
plt.figure(figsize=(10, 10))
for images, _ in train_ds.take(1):
    for i in range(9):
        augmented = data_augmentation(images)
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(np.array(augmented[0]).astype('uint8'))
        plt.axis('off')
plt.suptitle('Augmented Images (same source image, different transforms)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Performance optimisation — cache & prefetch
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds   = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

---
## Task 1 — Improved CNN from Scratch (BN + Dropout + Augmentation)

In [ ]:
def build_improved_cnn(input_shape=(224, 224, 3), num_classes=6):
    """
    Deeper CNN with:
      • Data Augmentation (built into the model)
      • Rescaling (normalise to [0, 1])
      • 4 Convolutional Blocks, each with BN + Dropout
      • 4 Dense layers with BN + Dropout
    """
    model = Sequential([
        # Input
        layers.Input(shape=input_shape),

        # ── Augmentation & Rescaling ─────────────────────────────────────
        layers.Lambda(data_augmentation),
        layers.Rescaling(1.0 / 255),

        # ── Conv Block 1 ─────────────────────────────────────────────────
        Conv2D(32, (3, 3), padding='same', activation=None),
        BatchNormalization(),
        Activation('relu'),
        Conv2D(32, (3, 3), padding='same', activation=None),
        BatchNormalization(),
        Activation('relu'),
        MaxPooling2D((2, 2)),
        Dropout(0.25),

        # ── Conv Block 2 ─────────────────────────────────────────────────
        Conv2D(64, (3, 3), padding='same', activation=None),
        BatchNormalization(),
        Activation('relu'),
        Conv2D(64, (3, 3), padding='same', activation=None),
        BatchNormalization(),
        Activation('relu'),
        MaxPooling2D((2, 2)),
        Dropout(0.25),

        # ── Conv Block 3 ─────────────────────────────────────────────────
        Conv2D(128, (3, 3), padding='same', activation=None),
        BatchNormalization(),
        Activation('relu'),
        Conv2D(128, (3, 3), padding='same', activation=None),
        BatchNormalization(),
        Activation('relu'),
        MaxPooling2D((2, 2)),
        Dropout(0.25),

        # ── Conv Block 4 ─────────────────────────────────────────────────
        Conv2D(256, (3, 3), padding='same', activation=None),
        BatchNormalization(),
        Activation('relu'),
        MaxPooling2D((2, 2)),
        Dropout(0.25),

        # ── Classifier Head ──────────────────────────────────────────────
        Flatten(),

        Dense(512, activation=None),
        BatchNormalization(),
        Activation('relu'),
        Dropout(0.5),

        Dense(256, activation=None),
        BatchNormalization(),
        Activation('relu'),
        Dropout(0.5),

        Dense(128, activation=None),
        BatchNormalization(),
        Activation('relu'),
        Dropout(0.5),

        # ── Output ───────────────────────────────────────────────────────
        Dense(num_classes, activation='softmax'),
    ])
    return model


cnn_model = build_improved_cnn(input_shape=(224, 224, 3), num_classes=NUM_CLASSES)

cnn_model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cnn_model.summary()

In [ ]:
# ── Callbacks ────────────────────────────────────────────────────────────────
callbacks_cnn = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=10, restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=4, verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        'best_cnn_model.keras', monitor='val_accuracy',
        save_best_only=True, verbose=1
    ),
]

# ── Train ────────────────────────────────────────────────────────────────────
EPOCHS = 50   # EarlyStopping will halt training if val_loss stops improving

history_cnn = cnn_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks_cnn,
)

In [ ]:
# ── Plot Training History ─────────────────────────────────────────────────────
def plot_history(history, title='Model'):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Accuracy
    axes[0].plot(history.history['accuracy'],     label='Train Accuracy')
    axes[0].plot(history.history['val_accuracy'], label='Val Accuracy')
    axes[0].set_title(f'{title} — Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True)

    # Loss
    axes[1].plot(history.history['loss'],     label='Train Loss')
    axes[1].plot(history.history['val_loss'], label='Val Loss')
    axes[1].set_title(f'{title} — Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True)

    plt.tight_layout()
    plt.show()

plot_history(history_cnn, title='Task 1 — Custom CNN (BN + Dropout)')

In [ ]:
# ── Evaluate ─────────────────────────────────────────────────────────────────
cnn_loss, cnn_acc = cnn_model.evaluate(val_ds)
print(f"\nTask 1 — Custom CNN | Val Loss: {cnn_loss:.4f} | Val Accuracy: {cnn_acc:.4f}")

In [ ]:
# ── Inference & Classification Report ────────────────────────────────────────
y_true_cnn = []
y_pred_cnn = []

for images, labels in val_ds:
    preds = cnn_model.predict(images, verbose=0)
    y_pred_cnn.extend(np.argmax(preds, axis=1))
    y_true_cnn.extend(labels.numpy())

print("\nClassification Report — Task 1 (Custom CNN):")
print(classification_report(y_true_cnn, y_pred_cnn, target_names=class_names))

In [ ]:
# ── Visualise Predictions ─────────────────────────────────────────────────────
plt.figure(figsize=(12, 6))
sample_images, sample_labels = next(iter(val_ds))
preds = np.argmax(cnn_model.predict(sample_images[:9]), axis=1)

for i in range(9):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(sample_images[i].numpy().astype('uint8'))
    true_label = class_names[sample_labels[i]]
    pred_label = class_names[preds[i]]
    color = 'green' if true_label == pred_label else 'red'
    plt.title(f"True: {true_label}\nPred: {pred_label}", color=color, fontsize=8)
    plt.axis('off')

plt.suptitle('Task 1 Predictions  (green=correct, red=wrong)', fontsize=12)
plt.tight_layout()
plt.show()

---
## Task 2 — Transfer Learning with VGG16 (Fine-tuning)

In [ ]:
# ── Step 1: Load VGG16 base (ImageNet weights, no top) ───────────────────────
base_model = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)   # VGG16 requires at least 32×32; 224×224 is standard
)

print(f"VGG16 base model loaded. Total layers: {len(base_model.layers)}")
base_model.summary()

In [ ]:
# ── Step 2: Freeze all base-model layers ─────────────────────────────────────
for layer in base_model.layers:
    layer.trainable = False

trainable_count = sum(1 for l in base_model.layers if l.trainable)
print(f"Trainable layers in base model after freezing: {trainable_count}")

In [ ]:
# ── Step 3: Add augmentation + custom classification head ────────────────────
inputs = keras.Input(shape=(224, 224, 3))

# Augmentation (only active during training)
x = layers.RandomFlip('horizontal')(inputs)
x = layers.RandomRotation(0.15)(x)
x = layers.RandomZoom(0.1)(x)

# VGG16 preprocessing (scales to ImageNet statistics)
x = tf.keras.applications.vgg16.preprocess_input(x)

# Frozen VGG16 feature extractor
x = base_model(x, training=False)

# Custom head
x = GlobalAveragePooling2D()(x)

x = Dense(1024, activation=None)(x)
x = BatchNormalization()(x)
x = Activation('relu')(x)
x = Dropout(0.5)(x)

x = Dense(512, activation=None)(x)
x = BatchNormalization()(x)
x = Activation('relu')(x)
x = Dropout(0.4)(x)

outputs = Dense(NUM_CLASSES, activation='softmax')(x)

# ── Step 4: Create final model ────────────────────────────────────────────────
vgg_model = Model(inputs=inputs, outputs=outputs)

vgg_model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

vgg_model.summary()

In [ ]:
# ── Step 5: Train — Phase 1 (head only) ──────────────────────────────────────
callbacks_vgg = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=8, restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        'best_vgg16_model.keras', monitor='val_accuracy',
        save_best_only=True, verbose=1
    ),
]

print("Phase 1 — Training the custom head (VGG16 frozen)...")
history_vgg_phase1 = vgg_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks_vgg,
)

plot_history(history_vgg_phase1, title='Task 2 Phase 1 — VGG16 Head Only')

In [ ]:
# ── Optional Phase 2: Fine-tune last VGG16 block ─────────────────────────────
# Unfreeze block5 (last convolutional block) for fine-tuning
for layer in base_model.layers:
    if layer.name.startswith('block5'):
        layer.trainable = True

# Recompile with a much lower LR to avoid destroying pre-trained weights
vgg_model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Phase 2 — Fine-tuning VGG16 block5 + custom head...")
history_vgg_phase2 = vgg_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks_vgg,
)

plot_history(history_vgg_phase2, title='Task 2 Phase 2 — Fine-tuned VGG16 (block5)')

In [ ]:
# ── Evaluate ─────────────────────────────────────────────────────────────────
vgg_loss, vgg_acc = vgg_model.evaluate(val_ds)
print(f"\nTask 2 — VGG16 Transfer Learning | Val Loss: {vgg_loss:.4f} | Val Accuracy: {vgg_acc:.4f}")

In [ ]:
# ── Inference & Classification Report ────────────────────────────────────────
y_true_vgg = []
y_pred_vgg = []

for images, labels in val_ds:
    preds = vgg_model.predict(images, verbose=0)
    y_pred_vgg.extend(np.argmax(preds, axis=1))
    y_true_vgg.extend(labels.numpy())

print("\nClassification Report — Task 2 (VGG16 Transfer Learning):")
print(classification_report(y_true_vgg, y_pred_vgg, target_names=class_names))

In [ ]:
# ── Visualise Predictions ─────────────────────────────────────────────────────
plt.figure(figsize=(12, 6))
sample_images, sample_labels = next(iter(val_ds))
preds_vgg = np.argmax(vgg_model.predict(sample_images[:9]), axis=1)

for i in range(9):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(sample_images[i].numpy().astype('uint8'))
    true_label = class_names[sample_labels[i]]
    pred_label = class_names[preds_vgg[i]]
    color = 'green' if true_label == pred_label else 'red'
    plt.title(f"True: {true_label}\nPred: {pred_label}", color=color, fontsize=8)
    plt.axis('off')

plt.suptitle('Task 2 VGG16 Predictions  (green=correct, red=wrong)', fontsize=12)
plt.tight_layout()
plt.show()

---
## Comparison: Task 1 vs Task 2

In [ ]:
print("=" * 55)
print(f"{'Model':<35} {'Val Accuracy':>12} {'Val Loss':>8}")
print("=" * 55)
print(f"{'Task 1 — Custom CNN (BN+Dropout)':<35} {cnn_acc:>12.4f} {cnn_loss:>8.4f}")
print(f"{'Task 2 — VGG16 Transfer Learning':<35} {vgg_acc:>12.4f} {vgg_loss:>8.4f}")
print("=" * 55)

if vgg_acc > cnn_acc:
    improvement = (vgg_acc - cnn_acc) * 100
    print(f"\n✅ Transfer learning (VGG16) improved accuracy by {improvement:.2f} percentage points.")
else:
    diff = (cnn_acc - vgg_acc) * 100
    print(f"\n⚠️  Custom CNN matched or outperformed VGG16 by {diff:.2f} pp on this small dataset.")
    print("   Consider fine-tuning more layers or training for more epochs.")

In [ ]:
# ── Save the best model ───────────────────────────────────────────────────────
vgg_model.save('vgg16_fruits_classifier.keras')
cnn_model.save('custom_cnn_fruits_classifier.keras')
print("Both models saved successfully.")